# One process, 11,963 paths: purchase-to-pay process mining

Stephen tAMBI

Event log: BPI Challenge 2019 (van Dongen, 4TU.ResearchData), the purchase-to-pay process of a large multinational in coatings and paints. Convert the XES file first:

```
python ../scripts/xes_to_parquet.py ../data/BPI_Challenge_2019.xes.gz ../data/events.parquet
```

In [1]:
import pandas as pd, numpy as np
df = pd.read_parquet('../data/events.parquet').rename(columns={'case:concept:name': 'case', 'concept:name': 'act'})
df['ts'] = pd.to_datetime(df['time:timestamp'], utc=True, format='ISO8601')
df = df.sort_values(['case', 'ts'], kind='stable')
created = df[df.act == 'Create Purchase Order Item'].groupby('case').ts.min()
cases = created[(created >= '2018-01-01') & (created < '2019-01-01')].index
df = df[df.case.isin(cases)]
print(f'{len(df):,} events, {len(cases):,} order items, {df.act.nunique()} activities')

1,594,557 events, 251,352 order items, 42 activities


## Variants: the paths items really take

In [2]:
var = df.groupby('case').act.agg(' > '.join)
vc = var.value_counts()
print(f'{len(vc):,} variants; top variant {vc.iloc[0]/len(var):.1%}; {(vc.cumsum()/len(var) < .8).sum()+1} variants cover 80%; {(vc==1).sum():,} occur once')
(vc.head(5) / len(var) * 100).round(1)

11,963 variants; top variant 20.0%; 45 variants cover 80%; 9,025 occur once


act
Create Purchase Order Item > Vendor creates invoice > Record Goods Receipt > Record Invoice Receipt > Clear Invoice                                 20.0
Create Purchase Order Item > Record Goods Receipt > Vendor creates invoice > Record Invoice Receipt > Clear Invoice                                 12.3
Create Purchase Order Item > Record Goods Receipt                                                                                                    4.9
Create Purchase Order Item > Vendor creates invoice > Record Goods Receipt > Record Invoice Receipt > Remove Payment Block > Clear Invoice           4.5
Create Purchase Order Item > Receive Order Confirmation > Record Goods Receipt > Vendor creates invoice > Record Invoice Receipt > Clear Invoice     3.9
Name: count, dtype: float64

## Waiting times on the main path

In [3]:
first = lambda a, f='min': df[df.act == a].groupby('case').ts.agg(f)
def wait(a, b):
    d = (first(b) - first(a)).dropna().dt.total_seconds() / 86400
    return round(d[d >= 0].median(), 1)
for a, b in [('Create Purchase Order Item', 'Record Goods Receipt'), ('Record Goods Receipt', 'Record Invoice Receipt'), ('Record Invoice Receipt', 'Clear Invoice')]:
    print(f'{a} -> {b}: {wait(a, b)} days')

Create Purchase Order Item -> Record Goods Receipt: 10.0 days


Record Goods Receipt -> Record Invoice Receipt: 11.3 days


Record Invoice Receipt -> Clear Invoice: 42.1 days


## Order-to-payment time: clean vs payment block vs rework

In [4]:
paid = first('Clear Invoice', 'max')
ct = ((paid - created.reindex(paid.index)).dt.total_seconds() / 86400).pipe(lambda s: s[s >= 0])
blocked = set(df[df.act == 'Remove Payment Block'].case)
REWORK = ['Change Price', 'Change Quantity', 'Change Approval for Purchase Order', 'Cancel Invoice Receipt', 'Delete Purchase Order Item', 'Cancel Goods Receipt', 'Change Delivery Indicator', 'Cancel Subsequent Invoice', 'Block Purchase Order Item', 'Reactivate Purchase Order Item', 'Change Currency', 'Change payment term']
rework = set(df[df.act.isin(REWORK)].case)
groups = {'all': ct, 'payment block': ct[ct.index.isin(blocked)], 'rework': ct[ct.index.isin(rework)], 'clean': ct[~ct.index.isin(blocked | rework)]}
pd.DataFrame({k: {'items': len(v), 'median days': round(v.median(), 1), '% over 90 days': round((v > 90).mean() * 100, 1)} for k, v in groups.items()}).T

,items,median days,% over 90 days
all,183627.0,77.6,38.5
payment block,50551.0,87.0,48.8
rework,27519.0,93.9,54.5
clean,116458.0,71.2,32.5


## Purchase orders created after the invoice

In [5]:
vi = first('Vendor creates invoice')
after = vi < created.reindex(vi.index)
print(f'{after.sum():,} items ({after.sum()/len(cases):.2%}) got their PO after the vendor invoiced')

3,601 items (1.43%) got their PO after the vendor invoiced


## Who does the work

In [6]:
kind = np.where(df['org:resource'].str.startswith('user'), 'people', np.where(df['org:resource'].str.startswith('batch'), 'automated jobs', 'vendor or system'))
print(pd.Series(kind).value_counts(normalize=True).mul(100).round(1))
manual = df[kind == 'people']
print(f'{len(manual)/len(cases):.2f} manual steps per item')
manual.act.value_counts().head(6)

people              65.2
vendor or system    25.0
automated jobs       9.8
Name: proportion, dtype: float64


4.14 manual steps per item


act
Create Purchase Order Item          232129
Record Goods Receipt                229030
Record Invoice Receipt              219190
Clear Invoice                       186402
Remove Payment Block                 41694
Create Purchase Requisition Item     36355
Name: count, dtype: int64

## Recommendations
1. Find out why invoices get blocked (22% of items, +16 days).
2. Fix orders before they are sent (18% reworked).
3. Flag purchase orders created after the invoice.
4. Automate invoice matching for clean three-way matches.

Interactive version: https://stambi292.github.io/process-mining/